In [1]:
import tensorflow as tf

In [2]:
IMAGE_SIZE=224
CHANNELS=3
BATCH_SIZE=32

In [3]:
dataset= tf.keras.preprocessing.image_dataset_from_directory(
    "lichen_images",
    shuffle=True,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

Found 1824 files belonging to 10 classes.


In [4]:
class_names=dataset.class_names

In [ ]:
def partition_of_dataset(ds, train_split=0.8, val_split=0.1, shuffle=True, shuffle_size=1000):
    ds_size = len(ds)
    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=12)
    
    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)
    
    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(train_size).skip(val_size)
    
    return train_ds, val_ds, test_ds


In [6]:
# Split the data and apply performance optimizations
train_ds, val_ds, test_ds = partition_of_dataset(dataset)
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [7]:
from tensorflow import keras
from tensorflow.keras import models,layers
from tensorflow.keras.applications import EfficientNetB0

n_classes= 10
INPUT_SHAPE= (IMAGE_SIZE,IMAGE_SIZE,3)

data_augmentation= keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2)
],name="data_augmentation")

base_model= EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224,224,3),
)

model= keras.Sequential([
    layers.Input(shape=INPUT_SHAPE),
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(64,activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(n_classes,activation="softmax"),
])

In [8]:
INITIAL_EPOCHS = 25
FINE_TUNE_EPOCHS = 25
TOTAL_EPOCHS = INITIAL_EPOCHS + FINE_TUNE_EPOCHS

In [9]:
print("--- Starting Phase 1: Initial Training ---")
# Freeze the base model so only the new layers are trained
base_model.trainable = False

# Compile the model with a standard learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Define the learning rate scheduler callback
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7
)

# Train the model
history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    callbacks=[lr_scheduler]
)


--- Starting Phase 1: Initial Training ---
Epoch 1/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.1842 - loss: 2.2458 - val_accuracy: 0.5375 - val_loss: 1.6292 - learning_rate: 0.0010
Epoch 2/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 72s 2s/step - accuracy: 0.3765 - loss: 1.7717 - val_accuracy: 0.6187 - val_loss: 1.2677 - learning_rate: 0.0010
Epoch 3/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 72s 2s/step - accuracy: 0.4359 - loss: 1.5579 - val_accuracy: 0.6438 - val_loss: 1.1131 - learning_rate: 0.0010
Epoch 4/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.4722 - loss: 1.4372 - val_accuracy: 0.7000 - val_loss: 1.0210 - learning_rate: 0.0010
Epoch 5/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.5402 - loss: 1.3125 - val_accuracy: 0.7188 - val_loss: 0.9518 - learning_rate: 0.0010
Epoch 6/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.5540 - loss: 1.2838 - val_accuracy: 0.7250 - val_loss: 0.9250 - learning_rate: 0.0010
Epoch 7/30
45/45 ━━━━━━━━━━━━━━━━━━━━ 72s 2s/step - accu

In [10]:
model.save('true_model_version_1.keras')

In [12]:
print("\n--- Evaluating Final Model ---")
final_loss, final_accuracy = model.evaluate(test_ds)
print(f"Final Test Loss: {final_loss:.4f}")
print(f"Final Test Accuracy: {final_accuracy*100:.2f}%")


--- Evaluating Final Model ---
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.7088 - loss: 0.7478
Final Test Loss: 0.7805
Final Test Accuracy: 70.54%
